In [1]:
import numpy as np 
import pandas as pd
import re
import gc
import os
import fileinput
import string
import tensorflow.compat.v1 as tf
import zipfile
import datetime
import sys
from tqdm  import tqdm
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score, roc_auc_score

from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Flatten, LSTM, Conv1D, MaxPooling1D, Dropout, Activation
from keras.layers.embeddings import Embedding
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [2]:
tqdm.pandas()

In [3]:
valid_forms = ['am','are','were','was','is','been','being','be']
blank = '----'

In [4]:
def detect(tokens):
    return [t for t in tokens if t in valid_forms]
    
def replace_blank(tokens):
    return [blank if t in valid_forms else t for t in tokens]

def create_windows(tokens, window_size=3):
    X = []
    for i, word in enumerate(tokens):
        if word == blank:
            window = tokens[i-window_size:i] + tokens[i+1:i+window_size+1]
            window = ' '.join(window)
            X.append(window)    
    return X

In [5]:
df1 = pd.read_excel(r".\train.xlsm")
df1.set_index("#id", inplace = True)

In [6]:
df1.head()

,Text,Label
#id,,
1,Long-standing presence in industry coupled wit...,1
2,"The Company, being in the business of manufact...",1
3,The debt equity ratio has been low during the ...,1
4,Efficient working capital management coupled w...,1
5,The Company’s working capital cycle remained m...,1


In [7]:
one_hot = pd.get_dummies(df1["Label"])
df1.drop(['Label'],axis=1,inplace=True)
df1 = pd.concat([df1,one_hot],axis=1)
df1.head()

,Text,1,2,3,4
#id,,,,,
1,Long-standing presence in industry coupled wit...,1,0,0,0
2,"The Company, being in the business of manufact...",1,0,0,0
3,The debt equity ratio has been low during the ...,1,0,0,0
4,Efficient working capital management coupled w...,1,0,0,0
5,The Company’s working capital cycle remained m...,1,0,0,0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(df1["Text"].values, df1.drop(['Text'],axis=1).values, test_size=0.2, random_state=42)

In [9]:
vocabulary_size = 10000
tokenizer = Tokenizer(num_words= vocabulary_size)
tokenizer.fit_on_texts(X_train)
sequences = tokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(sequences, maxlen=50)

In [10]:
sequences = tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(sequences, maxlen=50)

In [11]:
model = Sequential()
model.add(Embedding(10000, 100, input_length=50))
model.add(Dropout(0.2))
model.add(Conv1D(64, 5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(100))
model.add(Dense(4, activation='softmax'))

In [12]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [13]:
model.fit(X_train, y_train,
                    batch_size=1024,
                    epochs=10,
                    verbose=1,
                    validation_split=0.1)

Epoch 1/10
2/2 [==============================] - 2s 562ms/step - loss: 1.3860 - accuracy: 0.2794 - val_loss: 1.3734 - val_accuracy: 0.3850
Epoch 2/10
2/2 [==============================] - 0s 233ms/step - loss: 1.3750 - accuracy: 0.3222 - val_loss: 1.3578 - val_accuracy: 0.3850
Epoch 3/10
2/2 [==============================] - 0s 224ms/step - loss: 1.3639 - accuracy: 0.3222 - val_loss: 1.3390 - val_accuracy: 0.3850
Epoch 4/10
2/2 [==============================] - 0s 236ms/step - loss: 1.3542 - accuracy: 0.3222 - val_loss: 1.3207 - val_accuracy: 0.3850
Epoch 5/10
2/2 [==============================] - 0s 215ms/step - loss: 1.3475 - accuracy: 0.3222 - val_loss: 1.3095 - val_accuracy: 0.3850
Epoch 6/10
2/2 [==============================] - 0s 235ms/step - loss: 1.3321 - accuracy: 0.3222 - val_loss: 1.3009 - val_accuracy: 0.3850
Epoch 7/10
2/2 [==============================] - 0s 220ms/step - loss: 1.3109 - accuracy: 0.3222 - val_loss: 1.2841 - val_accuracy: 0.4000
Epoch 8/10
2/2 [====

In [14]:
score = model.evaluate(X_test, y_test,
                       batch_size=256, verbose=1)
print('Test accuracy:', score[1])

2/2 [==============================] - 0s 16ms/step - loss: 1.1305 - accuracy: 0.4340
Test accuracy: 0.4339999854564667


In [15]:
preds = model.predict(X_test)

In [16]:
print(classification_report(np.argmax(y_test,axis=1),np.argmax(preds,axis=1)))

              precision    recall  f1-score   support

           0       0.42      0.91      0.57       170
           1       0.46      0.43      0.44       129
           2       0.86      0.06      0.11       105
           3       1.00      0.02      0.04        96

    accuracy                           0.43       500
   macro avg       0.68      0.35      0.29       500
weighted avg       0.63      0.43      0.34       500



In [17]:
gc.collect()

2423

# BERT implementation

In [18]:
#!wget https://storage.googleapis.com/bert_models/2018_10_18/uncased_L-12_H-768_A-12.zip
#!wget https://raw.githubusercontent.com/google-research/bert/master/modeling.py 
#!wget https://raw.githubusercontent.com/google-research/bert/master/optimization.py 
#!wget https://raw.githubusercontent.com/google-research/bert/master/run_classifier.py 
#!wget https://raw.githubusercontent.com/google-research/bert/master/tokenization.py 

In [45]:
tf.disable_v2_behavior()
import modeling
import optimization
import run_classifier
import tokenization

In [46]:
folder = 'model_folder'
with zipfile.ZipFile("uncased_L-12_H-768_A-12.zip","r") as zip_ref:
    zip_ref.extractall(folder)

In [47]:
BERT_MODEL = 'uncased_L-12_H-768_A-12'
BERT_PRETRAINED_DIR = f'{folder}/uncased_L-12_H-768_A-12'
OUTPUT_DIR = f'{folder}/outputs'
print(f'>> Model output directory: {OUTPUT_DIR}')
print(f'>>  BERT pretrained directory: {BERT_PRETRAINED_DIR}')

>> Model output directory: model_folder/outputs
>>  BERT pretrained directory: model_folder/uncased_L-12_H-768_A-12


In [48]:
df1 = pd.read_excel(r"train.xlsm")
df1.set_index("#id", inplace = True)
df1.head()
one_hot = pd.get_dummies(df1["Label"])
df1 = pd.concat([df1,one_hot],axis=1)
df1.head()

,Text,Label,1,2,3,4
#id,,,,,,
1,Long-standing presence in industry coupled wit...,1,1,0,0,0
2,"The Company, being in the business of manufact...",1,1,0,0,0
3,The debt equity ratio has been low during the ...,1,1,0,0,0
4,Efficient working capital management coupled w...,1,1,0,0,0
5,The Company’s working capital cycle remained m...,1,1,0,0,0


In [49]:
X_train, X_test, y_train, y_test = train_test_split(df1["Text"].values, df1["Label"].values, test_size=0.2, random_state=42)

In [50]:
y_train

array([1, 1, 2, ..., 4, 3, 4], dtype=int64)

In [51]:
def create_examples(lines, set_type, labels=None):
#Generate data for the BERT model
    guid = f'{set_type}'
    examples = []
    if guid == 'train':
        for line, label in zip(lines, labels):
            text_a = line
            label = str(label)
            examples.append(
              run_classifier.InputExample(guid=guid, text_a=text_a, text_b=None, label=label))
    else:
        for line in lines:
            text_a = line
            label = '0'
            examples.append(
              run_classifier.InputExample(guid=guid, text_a=text_a, text_b=None, label=label))
    return examples

In [52]:
# Model Hyper Parameters
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 3.0
WARMUP_PROPORTION = 0.1
MAX_SEQ_LENGTH = 50
# Model configs
SAVE_CHECKPOINTS_STEPS = 100000 #if you wish to finetune a model on a larger dataset, use larger interval
# each checpoint weights about 1,5gb
ITERATIONS_PER_LOOP = 100000
NUM_TPU_CORES = 8
VOCAB_FILE = os.path.join(BERT_PRETRAINED_DIR, 'vocab.txt')
CONFIG_FILE = os.path.join(BERT_PRETRAINED_DIR, 'bert_config.json')
INIT_CHECKPOINT = os.path.join(BERT_PRETRAINED_DIR, 'bert_model.ckpt')
DO_LOWER_CASE = BERT_MODEL.startswith('uncased')

label_list = [str(num) for num in range(8)]
tokenizer = tokenization.FullTokenizer(vocab_file=VOCAB_FILE, do_lower_case=DO_LOWER_CASE)
train_examples = create_examples(X_train, 'train', labels=y_train)

tpu_cluster_resolver = None #Since training will happen on GPU, we won't need a cluster resolver

In [53]:
#TPUEstimator also supports training on CPU and GPU. You don't need to define a separate tf.estimator.Estimator.
run_config = tf.compat.v1.estimator.tpu.RunConfig(
    cluster=tpu_cluster_resolver,
    model_dir=OUTPUT_DIR,
    save_checkpoints_steps=SAVE_CHECKPOINTS_STEPS,
    tpu_config=tf.compat.v1.estimator.tpu.TPUConfig(
        iterations_per_loop=ITERATIONS_PER_LOOP,
        num_shards=NUM_TPU_CORES,
        per_host_input_for_training=tf.compat.v1.estimator.tpu.InputPipelineConfig.PER_HOST_V2))

In [54]:
num_train_steps = int(
    len(train_examples) / TRAIN_BATCH_SIZE * NUM_TRAIN_EPOCHS)
num_warmup_steps = int(num_train_steps * WARMUP_PROPORTION)

In [55]:
model_fn = run_classifier.model_fn_builder(
    bert_config=modeling.BertConfig.from_json_file(CONFIG_FILE),
    num_labels=len(label_list),
    init_checkpoint=INIT_CHECKPOINT,
    learning_rate=LEARNING_RATE,
    num_train_steps=num_train_steps,
    num_warmup_steps=num_warmup_steps,
    use_tpu=False, #If False training will fall on CPU or GPU, depending on what is available  
    use_one_hot_embeddings=True)

In [56]:
estimator = tf.compat.v1.estimator.tpu.TPUEstimator(
    use_tpu=False, #If False training will fall on CPU or GPU, depending on what is available 
    model_fn=model_fn,
    config=run_config,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE)

INFO:tensorflow:Using config: {'_model_dir': 'model_folder/outputs', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': 100000, '_save_checkpoints_secs': None, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': None, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1, '_tpu_config': TPUConfig(iterations_per_loop=100000, num_shards=8, num_cores_per_replica=None, per_host_input_f

In [57]:
print('Please wait...')
train_features = run_classifier.convert_examples_to_features(
    train_examples, label_list, MAX_SEQ_LENGTH, tokenizer)
print('>> Started training at {} '.format(datetime.datetime.now()))
print('  Num examples = {}'.format(len(train_examples)))
print('  Batch size = {}'.format(TRAIN_BATCH_SIZE))
tf.compat.v1.logging.info("  Num steps = %d", num_train_steps)
train_input_fn = run_classifier.input_fn_builder(
    features=train_features,
    seq_length=MAX_SEQ_LENGTH,
    is_training=True,
    drop_remainder=True)
estimator.train(input_fn=train_input_fn, max_steps=num_train_steps)
print('>> Finished training at {}'.format(datetime.datetime.now()))

Please wait...
INFO:tensorflow:Writing example 0 of 2000
INFO:tensorflow:*** Example ***
INFO:tensorflow:guid: train
INFO:tensorflow:tokens: [CLS] established relationship with reputed customers and suppliers coupled with wide geographical reach [SEP]
INFO:tensorflow:input_ids: 101 2511 3276 2007 22353 6304 1998 20141 11211 2007 2898 10056 3362 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
INFO:tensorflow:input_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
INFO:tensorflow:segment_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
INFO:tensorflow:label: 1 (id = 1)
INFO:tensorflow:*** Example ***
INFO:tensorflow:guid: train
INFO:tensorflow:tokens: [CLS] ss ##l is led by chairman and managing director , mr . sub ##has ##h chan ##der kat ##hur ##ia , a bachelor in commerce , with more than three decades of experience in the field of manufacturing o

AttributeError: module 'tensorflow.compat.v1' has no attribute 'contrib'

In [ ]:
def input_fn_builder(features, seq_length, is_training, drop_remainder):
  """Creates an `input_fn` closure to be passed to TPUEstimator."""

  all_input_ids = []
  all_input_mask = []
  all_segment_ids = []
  all_label_ids = []

  for feature in features:
    all_input_ids.append(feature.input_ids)
    all_input_mask.append(feature.input_mask)
    all_segment_ids.append(feature.segment_ids)
    all_label_ids.append(feature.label_id)

  def input_fn(params):
    """The actual input function."""
    print(params)
    batch_size = 500

    num_examples = len(features)

    d = tf.data.Dataset.from_tensor_slices({
        "input_ids":
            tf.constant(
                all_input_ids, shape=[num_examples, seq_length],
                dtype=tf.int32),
        "input_mask":
            tf.constant(
                all_input_mask,
                shape=[num_examples, seq_length],
                dtype=tf.int32),
        "segment_ids":
            tf.constant(
                all_segment_ids,
                shape=[num_examples, seq_length],
                dtype=tf.int32),
        "label_ids":
            tf.constant(all_label_ids, shape=[num_examples], dtype=tf.int32),
    })

    if is_training:
      d = d.repeat()
      d = d.shuffle(buffer_size=100)

    d = d.batch(batch_size=batch_size, drop_remainder=drop_remainder)
    return d

  return input_fn

In [ ]:
predict_examples = create_examples(X_test, 'test')

predict_features = run_classifier.convert_examples_to_features(
    predict_examples, label_list, MAX_SEQ_LENGTH, tokenizer)

predict_input_fn = input_fn_builder(
    features=predict_features,
    seq_length=MAX_SEQ_LENGTH,
    is_training=False,
    drop_remainder=False)

result = estimator.predict(input_fn=predict_input_fn)

In [ ]:
preds = []
for prediction in result:
      preds.append(np.argmax(prediction['probabilities']))

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
y_test.shape

In [ ]:
from collections.abc import Sequence


def get_shape(lst, shape=()):
    """
    returns the shape of nested lists similarly to numpy's shape.

    :param lst: the nested list
    :param shape: the shape up to the current recursion depth
    :return: the shape including the current depth
            (finally this will be the full depth)
    """

    if not isinstance(lst, Sequence):
        # base case
        return shape

    # peek ahead and assure all lists in the next depth
    # have the same length
    if isinstance(lst[0], Sequence):
        l = len(lst[0])
        if not all(len(item) == l for item in lst):
            msg = 'not all lists have the same length'
            raise ValueError(msg)

    shape += (len(lst), )

    # recurse
    shape = get_shape(lst[0], shape)

    return shape

In [ ]:
print(get_shape(preds)) 

In [ ]:
print("Accuracy of BERT is:",accuracy_score(y_test,preds))

In [ ]:
print(classification_report(y_test,preds))

In [ ]:
BATCH_SIZE = 50
def serving_input_receiver_fn():
    feature_spec = {
        "unique_ids": tf.FixedLenFeature([], tf.int64),
        "input_ids": tf.FixedLenFeature([MAX_SEQ_LENGTH], tf.int64),
        "input_mask": tf.FixedLenFeature([MAX_SEQ_LENGTH], tf.int64),
        "segment_ids": tf.FixedLenFeature([MAX_SEQ_LENGTH], tf.int64),
        "label_ids": tf.FixedLenFeature([MAX_SEQ_LENGTH], tf.int64),
    }

    serialized_tf_example = tf.placeholder(dtype=tf.string,
                                           shape=[BATCH_SIZE],
                                           name='input_example_tensor')
    receiver_tensors = {'examples': serialized_tf_example}
    features = tf.parse_example(serialized_tf_example, feature_spec)
    return tf.estimator.export.ServingInputReceiver(features, receiver_tensors)

In [ ]:
EXPORT_PATH = r'C:\Test\SWOT\Training\BERT\saved_model'

In [ ]:
estimator._export_to_tpu = False  ## !!important to add this
estimator.export_saved_model(
    export_dir_base = EXPORT_PATH,
    serving_input_receiver_fn = serving_input_receiver_fn)

In [ ]:
# Load model from export directory, and make a predict function.
predict_fn = tf.contrib.predictor.from_saved_model(r'C:\Test\SWOT\Training\BERT\saved_model\1581317597')

In [ ]:
input_text = {
    "Entire of the company ’ s revenue is derived by undertaking the outsourced services for its group entities . although the billing for the services rendered is done on monthly basis , there are delays in receipt from its group entities fund",
    'the company operates in the pharmaceutical industry which is highly regulated and governed by quality norms , patent legislation ##s as well as pricing policies . though the company outs ##our ##ces the manufacturing activities , the manufacturers need to ensure compliance of regulatory norms at their facilities'
}

In [ ]:
predict_examples = create_examples(input_text, 'input')

predict_features = run_classifier.convert_examples_to_features(
    predict_examples, label_list, MAX_SEQ_LENGTH, tokenizer)

predict_input_fn = input_fn_builder(
    features=predict_features,
    seq_length=MAX_SEQ_LENGTH,
    is_training=False,
    drop_remainder=False)

result = estimator.predict(input_fn=predict_input_fn)

In [ ]:
preds = []
for prediction in result:
      preds.append(np.argmax(prediction['probabilities']))

In [ ]:
for p in preds:
    print(p)